<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session03-lab.ipynb" target="_blank" rel="noopener noreferrer" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open published copy in Google Colab</a>

**Reviewing local changes?** The button opens the published notebook. To use a revision downloaded from the local course preview, upload that downloaded `.ipynb` file to Colab, or open it in Jupyter/VS Code.

# Session 3: Biological Databases

**Lab notebook — ten investigations using human myoglobin (*MB*)**

A name is a starting point. Your task is to connect archived DNA, a reference transcript, a protein, structures and experimental evidence, checking at each step that the record answers your biological question.

Work through all ten investigations and answer the corresponding ten graded questions in Canvas. Some investigations use code and others use websites; all contribute to the assessed lab. Record your findings in the supplied notes cells. You only need to edit indicated parameters and paste sequence exports; no new Python programming is required.

The lab is planned for a three-hour session, including setup, a short break, checking answers and submission. The estimate will be checked against student experience. A score of 5/10 meets the quiz pass threshold. Some questions have several correct options: follow the Canvas instructions for scoring and deadlines.

This notebook needs internet access and a Python 3 kernel. [Course setup](https://course.bioinfo.se/setup.html) · [Session 3 reading](https://course.bioinfo.se/session03.html)

## Setup

Run these two cells first. The first installs Biopython and Requests only when missing. The second defines the retrieval helpers used later. If a website is temporarily unavailable, save the error and retry after a short pause; do not substitute an unrelated search result.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {"biopython": "Bio", "requests": "requests"}
missing = [package for package, module in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from io import StringIO
import json
import re
import time
import requests
from Bio import SeqIO
from Bio.Seq import Seq
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

http = requests.Session()
http.headers["User-Agent"] = "KB8029-Session03-EducationalLab/2.0"
retries = Retry(total=2, backoff_factor=1,
                status_forcelist=(429, 502, 503, 504), allowed_methods={"GET"})
http.mount("https://", HTTPAdapter(max_retries=retries))
efetch = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
retrieved = datetime.now(timezone.utc).isoformat(timespec="seconds")


def get_response(url, params=None):
    response = http.get(url, params=params, timeout=45)
    response.raise_for_status()
    return response


def taxonomy_id(record):
    return next((xref.split(":", 1)[1]
                 for feature in record.features if feature.type == "source"
                 for xref in feature.qualifiers.get("db_xref", [])
                 if xref.startswith("taxon:")), "not listed")


def sequence_digest(sequence):
    return sha256(str(sequence).upper().encode("ascii")).hexdigest()[:16]


def show_sequence(label, sequence):
    print(f"{label}: {len(sequence)} letters; SHA256 prefix {sequence_digest(sequence)}")
    print(str(sequence)[:60], "...", str(sequence)[-30:])


def require_text(value, label):
    value = value.strip()
    if not value:
        raise ValueError(f"Fill in {label} in the indicated input cell, then rerun this cell.")
    return value

print("Retrieval started (UTC):", retrieved)

## Question 1 — One archived record, three interfaces

The accession `X00371.1` identifies an archived human myoglobin genomic DNA record. Open it at **all three** interfaces: [NCBI Nucleotide](https://www.ncbi.nlm.nih.gov/nuccore/X00371.1), [ENA](https://www.ebi.ac.uk/ena/browser/view/X00371.1) and [DDBJ](https://getentry.ddbj.nig.ac.jp/getentry/na/X00371.1/?filetype=html).

Find the accession/version, organism, molecule description, length and annotated features on each website. The archive institutions exchange records, but their websites and file formats differ. Compare their fields with the parsed records below. A sequence digest is a compact check of the nucleotide string: identical digests support sequence agreement; they do not establish that separate experiments were performed.

In [ ]:
archive_accession = "X00371.1"
archive_sources = {
    "NCBI": (efetch, {"db": "nuccore", "id": archive_accession,
                      "rettype": "gb", "retmode": "text"}, "genbank"),
    "ENA": (f"https://www.ebi.ac.uk/ena/browser/api/embl/{archive_accession}",
            {"download": "false"}, "embl"),
    "DDBJ": (f"https://getentry.ddbj.nig.ac.jp/getentry/na/{archive_accession}/",
             {"filetype": "text", "format": "flatfile"}, "genbank"),
}
archive_records = {}
for interface, (url, params, file_format) in archive_sources.items():
    record = SeqIO.read(StringIO(get_response(url, params).text), file_format)
    if record.id != archive_accession:
        raise ValueError(f"Unexpected accession returned by {interface}: {record.id}")
    archive_records[interface] = record
    print(f"\n{interface} | {record.id} | {record.annotations.get('organism', 'not listed')} "
          f"| taxon {taxonomy_id(record)}")
    print("Description:", record.description)
    print("Feature types:", ", ".join(sorted({feature.type for feature in record.features})))
    show_sequence("Nucleotides", record.seq)

**Record your observations.** Which fields agree? What distinguishes this genomic record from a spliced reference transcript? What does viewing it through three interfaces tell you about the number of underlying experiments?

- Three record URLs and retrieval date:
- Accession/version, organism and length:
- Interpretation of the comparison:

## Question 2 — Connect the gene, reference transcript and protein

Search for **human myoglobin** in [NCBI Gene](https://www.ncbi.nlm.nih.gov/gene/) and [UniProtKB](https://www.uniprot.org/uniprotkb). Confirm the organism and official gene symbol. In NCBI Gene, find RefSeq **transcript variant 1** and follow it to its nucleotide record. Record the accession **including its version suffix** and linked RefSeq protein accession. Independently find the reviewed human UniProtKB entry.

Enter the versioned transcript accession and UniProt accession below. Keep the gene, transcript and protein URLs in your notes. The code translates the annotated coding sequence (CDS), not the untranslated regions.

In [ ]:
# STUDENT INPUT: replace empty strings using the records you found.
transcript_accession = ""
uniprot_accession = ""

In [ ]:
transcript_accession = require_text(transcript_accession, "transcript_accession")
uniprot_accession = require_text(uniprot_accession, "uniprot_accession")
if not re.fullmatch(r"NM_\d+\.\d+", transcript_accession):
    raise ValueError("Enter a RefSeq mRNA accession including its version suffix.")
time.sleep(0.5)
response = get_response(efetch, {"db": "nuccore", "id": transcript_accession,
                                 "rettype": "gb", "retmode": "text"})
transcript = SeqIO.read(StringIO(response.text), "genbank")
gene_symbols = {name for feature in transcript.features
                for name in feature.qualifiers.get("gene", [])}
if (transcript.id != transcript_accession or taxonomy_id(transcript) != "9606"
        or "MB" not in gene_symbols or "transcript variant 1" not in transcript.description.lower()):
    raise ValueError("Check that your accession is the versioned human MB reference transcript described as variant 1.")
cds = next(feature for feature in transcript.features if feature.type == "CDS")
protein_accession = cds.qualifiers["protein_id"][0]
coding_sequence = cds.extract(transcript.seq)
translation_table = int(cds.qualifiers.get("transl_table", ["1"])[0])
translated = str(coding_sequence.translate(table=translation_table, cds=True))
if translated != cds.qualifiers["translation"][0]:
    raise ValueError("Translation disagrees with its annotation; ask the teacher to check the record.")
entry = get_response(f"https://rest.uniprot.org/uniprotkb/{uniprot_accession}.json").json()
entry_genes = {gene.get("geneName", {}).get("value") for gene in entry.get("genes", [])}
if (entry["primaryAccession"] != uniprot_accession or entry["organism"]["taxonId"] != 9606
        or "MB" not in entry_genes or entry["entryType"] != "UniProtKB reviewed (Swiss-Prot)"):
    raise ValueError("Check that your UniProt accession is the reviewed human MB entry.")
print("Transcript:", transcript.id, "|", transcript.description)
print("Molecule type:", transcript.annotations["molecule_type"])
print("Transcript length:", len(transcript.seq), "bases")
print("CDS interval on transcript (1-based, inclusive):",
      int(cds.location.start) + 1, "to", int(cds.location.end))
print("CDS length including stop codon:", len(coding_sequence), "bases")
print("RefSeq protein:", protein_accession)
show_sequence("CDS translation (terminal stop excluded)", translated)
print("\nUniProt:", entry["primaryAccession"], "|", entry["entryType"])
show_sequence("UniProt amino-acid sequence", entry["sequence"]["value"])
print("RefSeq cross-references listed by UniProt:")
for link in entry.get("uniProtKBCrossReferences", []):
    if link["database"] == "RefSeq":
        print(json.dumps(link, ensure_ascii=False))

**Interpret before answering.** Compare the linked identifiers, lengths and digests. Does the transcript contain only the CDS, or also untranslated sequence? The terminal stop codon in a complete CDS is excluded from the amino-acid string here.

- Gene record and organism:
- Variant 2 transcript and linked protein accession/version:
- Reviewed UniProt accession:
- Sequence comparison and what it establishes:

## Question 3 — Retrieve the sequence that answers the question

Open the [UCSC Genome Browser](https://genome.ucsc.edu/cgi-bin/hgGateway), select **human GRCh38/hg38**, and search for your versioned variant 1 transcript. Show the **NCBI RefSeq** track and open that precise transcript, not another isoform or a neighbouring gene.

Record the transcript span, strand and assembly. Browser coordinates are 1-based and include both ends. The full browser window may include flanking sequence, so do not use its padded interval as the transcript span.

From the transcript details, follow **Genomic Sequence from assembly**. Export **one FASTA record per gene**, in the transcript's orientation (for this minus-strand transcript, reverse-complement/transcript orientation). Use the transcript's full span, no upstream/downstream flanks, no repeat masking, and all upper case. Retrieve two outputs:

1. **UTR exons + CDS exons + introns:** select 5′ UTR exons, CDS exons, 3′ UTR exons and introns.
2. **CDS exons only:** leave CDS exons selected; uncheck both UTR types and introns.

If offered the current browser interval or gene/transcript region, use the transcript region. Save both complete FASTA outputs and paste them below. The code reports sequence summaries; you decide which sequence answers a biological question.

**If browser export is temporarily unavailable:** first attempt the browser task, then set `use_ucsc_api = True` below. The official UCSC API fallback uses the NCBI RefSeq annotation and the hg38 assembly sequence to reconstruct the same transcript span and concatenated CDS exons. It selects your exact transcript accession, handles the minus strand, and prints its source URLs. This is programmatic reconstruction from official data, not a captured browser export. Record that route in your notes and still inspect the assembly, track and annotation.

In [ ]:
# STUDENT INPUT: paste the complete FASTA outputs between the triple quotes.
genomic_fasta = """
"""
cds_fasta = """
"""
# If the browser export is unavailable, read the fallback explanation in Question 3.
use_ucsc_api = False

In [ ]:
if globals().get("use_ucsc_api", False):
    annotation_url = "https://api.genome.ucsc.edu/getData/track"
    annotation_params = {"genome": "hg38", "track": "ncbiRefSeq", "chrom": "chr22",
                         "start": 35606763, "end": 35617329}
    annotation_response = get_response(annotation_url, annotation_params)
    rows = [row for row in annotation_response.json().get("ncbiRefSeq", [])
            if row["name"] == transcript_accession and row["name2"] == "MB"]
    if len(rows) != 1:
        raise ValueError("The fallback did not find exactly one requested MB transcript in this region.")
    model = rows[0]
    sequence_response = get_response("https://api.genome.ucsc.edu/getData/sequence", {
        "genome": "hg38", "chrom": model["chrom"],
        "start": model["txStart"], "end": model["txEnd"]})
    genomic_sequence = Seq(sequence_response.json()["dna"].upper())
    if len(genomic_sequence) != model["txEnd"] - model["txStart"]:
        raise ValueError("Assembly sequence length disagrees with annotation boundaries.")
    starts = [int(value) for value in model["exonStarts"].rstrip(",").split(",")]
    ends = [int(value) for value in model["exonEnds"].rstrip(",").split(",")]
    coding_parts = [genomic_sequence[max(start, model["cdsStart"]) - model["txStart"]:
                                     min(end, model["cdsEnd"]) - model["txStart"]]
                    for start, end in zip(starts, ends)
                    if max(start, model["cdsStart"]) < min(end, model["cdsEnd"])]
    cds_sequence = Seq("").join(coding_parts)
    if model["strand"] == "-":
        genomic_sequence = genomic_sequence.reverse_complement()
        cds_sequence = cds_sequence.reverse_complement()

    def fasta_output(label, sequence):
        header = (f">hg38_ncbiRefSeq_{transcript_accession}_{label} "
                  f"{model['chrom']}:{model['txStart'] + 1}-{model['txEnd']} "
                  f"strand={model['strand']} orientation=transcript source=UCSC_API")
        return header + "\n" + "\n".join(str(sequence)[i:i+60] for i in range(0, len(sequence), 60)) + "\n"

    genomic_fasta = fasta_output("UTRs_CDS_introns", genomic_sequence)
    cds_fasta = fasta_output("CDS_only", cds_sequence)
    print("Sequence retrieval route: official UCSC API reconstruction")
    print("Annotation source:", annotation_response.url)
    print("Assembly sequence source:", sequence_response.url)


def read_export(text, label):
    text = require_text(text, label)
    if not text.lstrip().startswith(">"):
        raise ValueError(f"{label} must contain a FASTA header starting with > and its sequence.")
    records = list(SeqIO.parse(StringIO(text), "fasta"))
    if len(records) != 1:
        raise ValueError(f"{label}: choose one FASTA record per gene; got {len(records)} records.")
    record = records[0]
    if not record.seq or not set(str(record.seq).upper()).issubset(set("ACGTN")):
        raise ValueError(f"{label}: expected DNA letters A, C, G, T (or N), not HTML or protein.")
    return record

export_genomic = read_export(genomic_fasta, "genomic_fasta")
export_cds = read_export(cds_fasta, "cds_fasta")
for label, record in (("UTRs + CDS + introns", export_genomic), ("CDS exons only", export_cds)):
    print(f"\n{label}\nHeader: {record.description}")
    show_sequence("Export", record.seq)
print("\nAnnotated RefSeq CDS (for comparison):")
show_sequence("Reference CDS", coding_sequence)
if len(export_cds.seq) % 3:
    raise ValueError("CDS length is not divisible by three. Recheck region, checkboxes and strand.")
export_translation = str(export_cds.seq.upper().translate(table=translation_table))
print("CDS export translated in supplied orientation (including any stop symbol):")
print(export_translation)
print("Reference CDS translation (terminal stop excluded):")
print(translated)

**Record and explain your result.** Compare both exports with the genomic transcript span, the RefSeq transcript length and its CDS. Which retains introns? Which suits examination of an intronic variant? Which suits translation? If the CDS differs from RefSeq, check strand and export settings first; reference transcripts can also contain documented differences from an assembly.

- Assembly, annotation track and versioned transcript:
- Transcript span, strand and coordinate convention:
- Export 1 settings, length and header:
- Export 2 settings, length and header:
- Explanation of the differences:

## Question 4 — Investigate the evidence behind an annotation

Return to the reviewed UniProtKB entry. Inspect **Subcellular location** and **Subunit structure**, then open the evidence attached to each. Also read the protein-existence field. These fields address different questions: whether a protein has been detected, where it is found and how its molecules are organised.

The cell below displays annotation text and its attached evidence identifiers. Follow the publication link for the location annotation, and the source-protein link for the subunit annotation. Check the organism of the source protein. Read the relevant record descriptions and publication abstract; the full Methods section is not required.

In [ ]:
print("Protein existence:", entry.get("proteinExistence", "not listed"))
print("UniProt entry:", f"https://www.uniprot.org/uniprotkb/{uniprot_accession}/entry")


def show_evidence(evidences):
    if not evidences:
        print("  No explicit evidence item in this field; inspect the website context.")
    for evidence in evidences:
        evidence_id = evidence.get("id", "")
        source = evidence.get("source", "")
        print("  Evidence:", evidence.get("evidenceCode", ""), source, evidence_id)
        if source == "PubMed" and evidence_id:
            print(f"  https://pubmed.ncbi.nlm.nih.gov/{evidence_id}/")
        elif source == "UniProtKB" and evidence_id:
            print(f"  https://www.uniprot.org/uniprotkb/{evidence_id}/entry")


for comment in entry.get("comments", []):
    if comment["commentType"] == "SUBUNIT":
        print("\nSUBUNIT STRUCTURE")
        for text in comment.get("texts", []):
            print(text["value"])
            show_evidence(text.get("evidences", []))
    elif comment["commentType"] == "SUBCELLULAR LOCATION":
        print("\nSUBCELLULAR LOCATION")
        for location in comment.get("subcellularLocations", []):
            for field, value in location.items():
                if isinstance(value, dict) and "value" in value:
                    print(field + ":", value["value"])
                    show_evidence(value.get("evidences", []))
        for text in comment.get("note", {}).get("texts", []):
            print(text["value"])
            show_evidence(text.get("evidences", []))

**Record two claims and their evidence.** Explain the evidence category and what its source supports. Does the protein-existence label alone support every annotation?

- Location claim, evidence category and publication:
- Subunit claim, evidence category, source entry and source organism:
- Difference between existence evidence and evidence for a specific annotation:

## Question 5 — Distinguish a family from a domain

From UniProtKB, follow the **InterPro** cross-references for **Myoglobin** and **Globin**. Open both descriptions. Then view their annotations on the human protein, including sequence coverage. Compare the meaning of each entry with the region to which it is assigned, rather than copying two accession numbers.

A family groups related proteins; a domain describes a structural/evolutionary unit that can occur in proteins from different families. Two matches covering much of the same short protein do not automatically mean that two separate physical domains sit one after another.

- Myoglobin accession, entry type and description:
- Globin accession, entry type and description:
- Sequence coverage of each annotation:
- How the two descriptions fit together:

Use these observations to answer Question 5.

## Question 6 — Compare experimental and predicted structures

Follow an experimental structure and a predicted model using the same protein accession. The code lists the UniProtKB PDB cross-references, checks how **3RGK** was determined, and retrieves the matching full-length AlphaFold DB model metadata.

Open [3RGK at RCSB PDB](https://www.rcsb.org/structure/3RGK) or the [same deposited entry at PDBj](https://pdbj.org/mine/summary/3rgk), and the AlphaFold DB page printed below. Identify the source organism and sequence represented. Record whether the coordinates derive from experimental measurements or computational prediction. Resolution, pLDDT and structural alignment are not part of this investigation.

In [ ]:
pdb_links = sorted({link["id"] for link in entry.get("uniProtKBCrossReferences", [])
                    if link["database"] == "PDB"})
print("Experimental PDB cross-references listed by UniProt:")
for pdb_link in pdb_links:
    print(f"  {pdb_link}: https://www.rcsb.org/structure/{pdb_link}")
pdb_record = get_response("https://data.rcsb.org/rest/v1/core/entry/3RGK").json()
print("\n3RGK title:", pdb_record["struct"]["title"])
print("Experimental method:", ", ".join(item["method"] for item in pdb_record.get("exptl", [])))
print("Primary citation PubMed ID:", pdb_record.get("rcsb_primary_citation", {}).get(
    "pdbx_database_id_PubMed", "not listed"))
af_records = get_response(f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_accession}").json()
prediction = next((record for record in af_records
                   if record["uniprotAccession"] == uniprot_accession
                   and record["uniprotStart"] == 1
                   and record["uniprotEnd"] == len(entry["sequence"]["value"])), None)
if prediction is None:
    raise ValueError("No matching full-length AlphaFold model; check the accession and ask the teacher.")
print("\nAlphaFold DB model:", prediction["entryId"])
print("Reference accession:", prediction["uniprotAccession"])
print("Reference sequence positions:", prediction["uniprotStart"], "to", prediction["uniprotEnd"])
print("Organism:", prediction.get("organismScientificName", "inspect record page"))
print("Record page:", f"https://alphafold.ebi.ac.uk/entry/{uniprot_accession}")
print("Predicted coordinate file:", prediction.get("pdbUrl", prediction.get("cifUrl", "inspect record page")))

**Compare provenance and sequence.** Does a shared UniProt accession guarantee an experimental construct identical to the reference sequence? Keep both record identifiers. The next investigation examines the construct more closely.

- Experimental entry, method and source organism:
- Predicted model and reference sequence accession:
- What the connection between the records establishes:
- What still needs checking:

## Question 7 — Choose an appropriate experimental structure

A colleague requests **an experimentally determined structure of the human reference myoglobin sequence without amino-acid substitutions**. Compare three candidates:

- **3RGK:** [RCSB](https://www.rcsb.org/structure/3RGK) · [PDBj](https://pdbj.org/mine/summary/3rgk)
- **1WLA:** [RCSB](https://www.rcsb.org/structure/1WLA) · [PDBj](https://pdbj.org/mine/summary/1wla)
- **1MBN:** [RCSB](https://www.rcsb.org/structure/1MBN) · [PDBj](https://pdbj.org/mine/summary/1mbn)

These form an instructor-selected comparison set. They are **not** all cross-references from the human UniProt entry. Inspect source organism, expression host, sequence mapping, mutation information and bound molecules. An expression host produces the experimental sample; it need not be the source species of the protein.

The code collects deposited metadata and complete entity sequences. Use the pages to interpret it and decide which candidates, if any, meet the request. The decision concerns these three candidates, not whether a suitable structure exists anywhere in the PDB. No structural superposition or numerical structure-quality assessment is needed.

In [ ]:
comparison_pdb_ids = ["3RGK", "1WLA", "1MBN"]


def organism_names(items):
    return "; ".join(sorted({item.get("scientific_name", "not listed") for item in items})) or "not listed"


structure_comparison = {}
for pdb_id in comparison_pdb_ids:
    metadata = get_response(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}").json()
    print(f"\n{'=' * 60}\n{pdb_id}: {metadata['struct']['title']}")
    print("Method:", ", ".join(item["method"] for item in metadata.get("exptl", [])))
    print("Pages:", f"https://www.rcsb.org/structure/{pdb_id}",
          f"https://pdbj.org/mine/summary/{pdb_id.lower()}")
    entities = []
    for entity_id in metadata["rcsb_entry_container_identifiers"].get("polymer_entity_ids", []):
        polymer = get_response(f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}").json()
        entities.append(polymer)
        print("Polymer entity:", entity_id)
        print("  Source organism:", organism_names(polymer.get("rcsb_entity_source_organism", [])))
        print("  Expression host:", organism_names(polymer.get("rcsb_entity_host_organism", [])))
        print("  UniProt mappings:", polymer.get("rcsb_polymer_entity_container_identifiers", {}).get(
            "uniprot_ids", []))
        print("  Mutation description:", polymer.get("rcsb_polymer_entity", {}).get(
            "pdbx_mutation", "not listed"))
        print("  Mutation count:", polymer.get("entity_poly", {}).get("rcsb_mutation_count", "not listed"))
        sequence = polymer.get("entity_poly", {}).get("pdbx_seq_one_letter_code_can", "")
        if sequence:
            show_sequence("  Deposited polymer sequence", "".join(sequence.split()))
    print("Bound nonpolymer components:")
    for entity_id in metadata["rcsb_entry_container_identifiers"].get("non_polymer_entity_ids", []):
        component = get_response(f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{entity_id}").json()
        molecule = component.get("pdbx_entity_nonpoly", {})
        print(" ", molecule.get("comp_id", "not listed"), molecule.get("name", "inspect record page"))
    structure_comparison[pdb_id] = {"entry": metadata, "entities": entities}

**Make a justified selection.** Record one reason for accepting or rejecting each candidate. A structure can still suit a different question even when it does not meet this request.

- 3RGK: source species, expression host, sequence/construct and decision:
- 1WLA: source species, expression host, sequence/construct and decision:
- 1MBN: source species, expression host, sequence/construct and decision:
- Bound molecule(s) recorded for the candidates:
- Your conclusion and its limits:

## Question 8 — Follow an interaction to its experiment

Return to the human myoglobin UniProtKB entry and open **Interaction**. Follow the **IntAct** evidence for its association with **cortistatin** from the **2021** study. If a participant page opens, find the pair in the interaction list and open its experimental evidence. More than one study can be listed: use the year and pair to select the requested evidence.

Record the protein accessions, experimental roles (bait/prey), detection method, host cell system, interaction-evidence accession and PubMed identifier. Follow the publication cross-reference and check its title, authors and abstract. A participant accession identifies a molecule; an interaction-evidence accession identifies a particular evidence record. Keep them distinct.

- Myoglobin and cortistatin accessions:
- IntAct participant URL and specific evidence accession/URL:
- Bait and prey:
- Method and cell system:
- Publication title, year and PubMed identifier:

Use these observations to answer Question 8.

**Alternative access route if the IntAct website is unavailable.** The official [IntAct pair export](https://www.ebi.ac.uk/Tools/webservices/psicquic/intact/webservices/current/search/query/id:P02144%20AND%20id:O00230?format=tab27) contains the experimental records in the standard MITAB format. Set the flag below to display selected fields with readable labels. Compare the publication years and identify the 2021 evidence; record that you used the export rather than the graphical interface. This is an alternative route for the same required investigation.

In [ ]:
use_intact_api = False  # Set True only if you need the alternative access route.
if use_intact_api:
    interaction_response = get_response(
        "https://www.ebi.ac.uk/Tools/webservices/psicquic/intact/webservices/current/"
        "search/query/id:P02144%20AND%20id:O00230", {"format": "tab27"})
    fields_to_show = {0: "Protein A", 1: "Protein B", 6: "Detection method",
                      7: "First author and year", 8: "Publication", 11: "Interaction type",
                      13: "Interaction evidence accession", 18: "Experimental role A",
                      19: "Experimental role B", 28: "Host system"}
    rows = [line.split("\t") for line in interaction_response.text.splitlines()
            if line.strip() and not line.startswith("#")]
    if not rows or any(len(row) < 29 for row in rows):
        raise ValueError("The IntAct service did not return a readable MITAB interaction export.")
    print("Source:", interaction_response.url)
    for row in rows:
        print("\nEXPERIMENTAL RECORD")
        for index, label in fields_to_show.items():
            print(f"{label}: {row[index]}")
else:
    print("Use the IntAct record pages for this investigation; the API export is an alternative route.")

## Question 9 — Decide what the evidence supports

Use the IntAct evidence from Question 8 and the linked publication's abstract. You do **not** need to read the full paper or its full Methods section.

In this kind of affinity-purification experiment, a tagged bait is isolated from a cell preparation. Other proteins retained with it can be detected by mass spectrometry.

Consider a student's claim: **“This study proves that myoglobin binds cortistatin directly in human skeletal muscle.”** Separate the observation from its interpretation. Which parts of that claim are addressed by the recorded method and cell system, and which would require further evidence? Use the source records to judge the Canvas options.

- What was detected, under which conditions:
- What the result supports:
- One stronger claim not established by this evidence alone:
- Sources used for your judgement:

## Question 10 — Make the browser view traceable

A classmate wants to locate **the same human myoglobin transcript and genomic interval** in UCSC. They need to distinguish the assembly sequence from the annotation placed on it, and the transcript span from any wider display window.

Complete the log below using Question 3. Reopen your saved browser URL and check the assembly, versioned transcript and interval against it. This checks record-keeping, not performance of a new experiment. The Canvas question asks you to diagnose an incomplete example log.

In [ ]:
# STUDENT INPUT: complete this log and run the cell to display your notes.
browser_log = {
    "organism": "",
    "assembly_name_and_browser_id": "",
    "annotation_track": "",
    "transcript_accession_with_version": "",
    "transcript_chromosome_interval": "",
    "transcript_strand": "",
    "coordinate_convention": "",
    "browser_view_url": "",
    "sequence_retrieval_route": "",
    "export1_regions_flanks_orientation": "",
    "export2_regions_flanks_orientation": "",
    "fasta_headers_and_lengths": "",
    "retrieved_utc": retrieved,
}
print(json.dumps(browser_log, indent=2, ensure_ascii=False))

## Review and submit

Check that all ten investigations and ten graded questions are complete. Keep this notebook, the browser log and two FASTA exports. A score of 5/10 meets the pass threshold; consult the course rules for deadlines and redo policy.

Submit the Canvas quiz and complete **Session 3 Lab: feedback** by the same deadline. The survey records completion while keeping responses anonymous. Both submissions are needed for the on-time lab bonus. Report investigation time separately from setup or website outages; this helps calibrate the three-hour lab.